# NUMARI — Agentes de Búsqueda
**Introducción a la Inteligencia Artificial · Prof. Carlos B. Ogando M.**

Este notebook implementa el juego Numari jugable por consola y tres agentes de búsqueda:
- **DFS** — Depth-First Search (pila explícita)
- **BFS** — Breadth-First Search (cola explícita)
- **A\*** — A-Star Search (cola de prioridad, 5 heurísticas normalizadas)

Los niveles se leen desde archivos `.txt` (formato: `.` = vacía, número = pasos).


## 0. Setup — Crear directorios y niveles


In [ ]:
import os
os.makedirs('niveles', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Formato: '.' = vacía, número = pasos
# Fáciles   → 3x3 (01-02) y 4x4 (03-04)
# Intermedios → 5x5 (05-06) y 6x6 (07-08)
# Difíciles  → 7x7 (09-10) y 8x8 (11-12)
# Muy difíciles → 10x10 (13-16)
NIVELES_TXT = {
    # ── Fáciles: 3x3 (verificados resolubles) ────────────
    'nivel_01': '2 2 .\n. . .\n. . 2\n',
    'nivel_02': '2 . 2\n. . .\n. 2 .\n',
    # ── Fáciles: 4x4 (verificados resolubles) ────────────
    'nivel_03': '3 . . .\n. . . 3\n3 . . .\n. . . 3\n',
    'nivel_04': '. 3 . 3\n. . . .\n. . . .\n3 . 3 .\n',
    # ── Intermedios: 5x5 (verificados resolubles) ────────
    'nivel_05': '4 . . . .\n. . . . 4\n. . 4 . .\n4 . . . .\n. . . . 4\n',
    'nivel_06': '4 . . 4 .\n. . . . .\n. . 4 . .\n4 . . . .\n. . . . 4\n',
    # ── Intermedios: 6x6 (verificados resolubles) ────────
    'nivel_07': '5 . . . . .\n. . . . . 5\n5 . . . . .\n. . . . . 5\n5 . . . . .\n. . . . . 5\n',
    'nivel_08': '5 . . . . .\n. . . . . 5\n. 5 . . 5 .\n. 5 . . 5 .\n5 . . . . .\n. . . . . 5\n',
    # ── Difíciles: 7x7 ───────────────────────────────────
    'nivel_09': '6 . . . . . .\n. . . . . . 6\n6 . . . . . .\n. . . . . . 6\n6 . . . . . .\n. . . . . . 6\n6 . . . . . .\n',
    'nivel_10': '. 6 . . . . .\n. . . . . 6 .\n6 . . . . . .\n. . . 6 . . .\n. . . . . . 6\n6 . . . . . .\n. . . 6 . . .\n',
    # ── Difíciles: 8x8 ───────────────────────────────────
    'nivel_11': '3 . . . 3 . . .\n. . . 3 . . . 3\n3 . . . . . 3 .\n. . 3 . . 3 . .\n. 3 . . . . . 3\n3 . . . 3 . . .\n. . . 3 . . 3 .\n. 3 . . . 3 . .\n',
    'nivel_12': '. 3 . 3 . . . .\n3 . . . . 3 . .\n. . 3 . . . 3 .\n. 3 . . 3 . . .\n3 . . . . . . 3\n. . 3 . . 3 . .\n. 3 . . 3 . . .\n3 . . 3 . . 3 .\n',
    # ── Muy difíciles: 10x10 ─────────────────────────────
    'nivel_13': '4 . . . . 4 . . . .\n. . . . 4 . . . . 4\n4 . . . . . . 4 . .\n. . 4 . . . 4 . . .\n. 4 . . . . . . 4 .\n4 . . . 4 . . . . .\n. . . 4 . . . . 4 .\n. 4 . . . . 4 . . .\n4 . . . . 4 . . . .\n. . . . 4 . . . 4 .\n',
    'nivel_14': '. 4 . . 4 . . 4 . .\n4 . . 4 . . 4 . . 4\n. . 4 . . 4 . . 4 .\n. 4 . . 4 . . 4 . .\n4 . . 4 . . 4 . . 4\n. . 4 . . 4 . . 4 .\n. 4 . . 4 . . 4 . .\n4 . . 4 . . 4 . . 4\n. . 4 . . 4 . . 4 .\n. 4 . . 4 . . 4 . .\n',
    'nivel_15': '4 . . . 4 . . . 4 .\n. . 4 . . . 4 . . .\n. 4 . . . 4 . . . 4\n4 . . 4 . . . 4 . .\n. . . . 4 . . . . 4\n4 . 4 . . . 4 . . .\n. . . . . 4 . . 4 .\n. 4 . . 4 . . . . 4\n4 . . 4 . . 4 . . .\n. . 4 . . . . 4 . 4\n',
    'nivel_16': '. . 4 . . . 4 . . 4\n4 . . . 4 . . 4 . .\n. 4 . . . . . . 4 .\n. . . 4 . 4 . . . .\n4 . . . . . . . 4 .\n. 4 . 4 . . 4 . . .\n. . . . 4 . . . . 4\n4 . 4 . . . . 4 . .\n. . . . . 4 . . 4 .\n. 4 . . 4 . . . . 4\n',
}

for nombre, contenido in NIVELES_TXT.items():
    with open(f'niveles/{nombre}.txt', 'w') as f:
        f.write(contenido)

print('Niveles creados:', list(NIVELES_TXT.keys()))


## 1. Motor Base — Celda y Tablero


In [ ]:
import copy

DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right (UDLR)


class Celda:
    """
    Representa una casilla del tablero.
    valor = 0  → vacía
    valor > 0  → número (pasos a caminar)
    """
    def __init__(self, valor=0):
        self.valor    = valor
        self.visitada = False

    def es_numero(self): return self.valor > 0
    def es_vacia(self):  return self.valor == 0

    def __str__(self):
        if not self.visitada:
            return f" {self.valor} " if self.es_numero() else " . "
        return f"[{self.valor}]" if self.es_numero() else " * "

    def __repr__(self): return str(self)


class Tablero:
    """
    Estado completo del juego Numari.
    Puede construirse desde archivo .txt o lista 2D.
    """

    def __init__(self):
        self.celdas    = []
        self.filas     = 0
        self.cols      = 0
        self.pos       = None
        self.pasos     = 0
        self.visitadas = 0
        self.total     = 0

    @classmethod
    def desde_archivo(cls, ruta):
        with open(ruta, 'r', encoding='utf-8') as f:
            lineas = [l.strip() for l in f if l.strip()]
        grid = []
        for linea in lineas:
            fila = [0 if tok == '.' else int(tok) for tok in linea.split()]
            grid.append(fila)
        return cls.desde_grid(grid)

    @classmethod
    def desde_grid(cls, grid):
        t        = cls()
        t.filas  = len(grid)
        t.cols   = len(grid[0])
        t.total  = t.filas * t.cols
        t.celdas = [[Celda(v) for v in fila] for fila in grid]
        return t

    def clonar(self):
        return copy.deepcopy(self)

    def get_celda(self, r, c):
        return self.celdas[r][c]

    def dentro_de_rango(self, r, c):
        return 0 <= r < self.filas and 0 <= c < self.cols

    def adyacentes(self, r, c):
        return [(r+dr, c+dc) for dr, dc in DIRS
                if self.dentro_de_rango(r+dr, c+dc)]

    def movimientos_validos(self):
        """
        Retorna lista de (r,c) válidos en orden UDLR.
        - pos es None → cualquier número no visitado del tablero
        - pasos > 0   → vacías adyacentes no visitadas
        - pasos == 0  → números adyacentes no visitados
        """
        if self.pos is None:
            return [(r, c)
                    for r in range(self.filas)
                    for c in range(self.cols)
                    if self.celdas[r][c].es_numero()
                    and not self.celdas[r][c].visitada]
        r, c      = self.pos
        resultado = []
        for nr, nc in self.adyacentes(r, c):
            celda = self.get_celda(nr, nc)
            if celda.visitada:
                continue
            if self.pasos > 0 and celda.es_vacia():
                resultado.append((nr, nc))
            if self.pasos == 0 and celda.es_numero():
                resultado.append((nr, nc))
        return resultado

    def seleccionar(self, r, c):
        if not self.dentro_de_rango(r, c):
            raise ValueError('Coordenadas fuera del tablero.')
        celda = self.get_celda(r, c)
        if celda.visitada:
            raise ValueError('Esa celda ya fue visitada.')

        if self.pos is None:
            if not celda.es_numero():
                raise ValueError('Debes empezar en una celda con número.')
            celda.visitada  = True
            self.visitadas += 1
            self.pos        = (r, c)
            self.pasos      = celda.valor
            return f'Número {celda.valor} — camina {celda.valor} celda(s) vacía(s).'

        if (r, c) not in self.adyacentes(*self.pos):
            raise ValueError('Solo puedes moverte a celdas adyacentes.')

        if self.pasos > 0:
            if not celda.es_vacia():
                raise ValueError(f'Debes caminar por celdas vacías. Faltan {self.pasos} paso(s).')
            celda.visitada  = True
            self.visitadas += 1
            self.pos        = (r, c)
            self.pasos     -= 1
            return 'Selecciona el número adyacente.' if self.pasos == 0 else f'Quedan {self.pasos} paso(s).'
        else:
            if not celda.es_numero():
                raise ValueError('Selecciona el número adyacente para continuar.')
            celda.visitada  = True
            self.visitadas += 1
            self.pos        = (r, c)
            self.pasos      = celda.valor
            return f'Número {celda.valor} — camina {celda.valor} celda(s) vacía(s).'

    def esta_completo(self):
        return self.visitadas == self.total

    def sin_salida(self):
        return (self.pos is not None
                and not self.esta_completo()
                and len(self.movimientos_validos()) == 0)

    def estado(self):
        """Tupla hashable para detectar estados repetidos en búsqueda."""
        return (
            self.pos,
            self.pasos,
            tuple(self.celdas[r][c].visitada
                  for r in range(self.filas)
                  for c in range(self.cols))
        )

    def mostrar(self):
        encabezado = '      ' + '  '.join(str(j) for j in range(self.cols))
        print(encabezado)
        print('     ' + '-' * (self.cols * 4))
        for i, fila in enumerate(self.celdas):
            fila_str = ''.join(' ★ ' if self.pos == (i, j) else str(c)
                               for j, c in enumerate(fila))
            print(f'  {i}  | {fila_str}')
        print()


print('Motor base cargado OK')


## 2. Agentes de búsqueda


In [ ]:
import time
import tracemalloc
import heapq
from collections import deque


class Nodo:
    """Nodo genérico para DFS y BFS."""
    __slots__ = ('tablero', 'camino', 'profundidad')

    def __init__(self, tablero, camino, profundidad):
        self.tablero     = tablero
        self.camino      = camino
        self.profundidad = profundidad


class NodoAStar(Nodo):
    """Nodo extendido con costos g, h, f para A*."""
    __slots__ = ('tablero', 'camino', 'profundidad', 'g', 'h', 'f')

    def __init__(self, tablero, camino, profundidad, g, h):
        super().__init__(tablero, camino, profundidad)
        self.g = g
        self.h = h
        self.f = g + h


class Agente:
    """
    Clase base para los agentes de búsqueda.
    Define la interfaz común y los helpers de medición y output.
    """

    nombre = 'Agente'

    def resolver(self, tablero_inicial, output_path):
        raise NotImplementedError

    def _verificar_solucion(self, tablero_inicial, camino):
        """Reproduce el camino sobre copia limpia y confirma que completa el tablero."""
        t = tablero_inicial.clonar()
        try:
            for r, c in camino:
                t.seleccionar(r, c)
        except ValueError:
            return False
        return t.esta_completo()

    def _iniciar_medicion(self):
        if tracemalloc.is_tracing():
            tracemalloc.stop()
        tracemalloc.start()
        return time.time()

    def _terminar_medicion(self, t_inicio):
        t_fin   = time.time()
        memoria = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        return round(t_fin - t_inicio, 6), round(memoria[1] / (1024 * 1024), 4)

    def _escribir_output(self, m, ruta):
        os.makedirs(os.path.dirname(ruta) if os.path.dirname(ruta) else '.', exist_ok=True)
        with open(ruta, 'w', encoding='utf-8') as f:
            for k, v in m.items():
                if k == 'path_to_goal':
                    coords = ' -> '.join(f'({r},{c})' for r, c in v) if v else 'Sin solucion'
                    f.write(f'{k:<22}: {coords}\n')
                else:
                    f.write(f'{k:<22}: {v}\n')

    def _construir_resultado(self, resultado, nodes_expanded, max_search_depth, running_time, max_ram, **extra):
        """Construye el diccionario de métricas estándar."""
        m = {
            'algoritmo'          : self.nombre,
            'solucion_encontrada': resultado is not None,
            'path_to_goal'       : resultado.camino      if resultado else [],
            'cost_of_path'       : len(resultado.camino) if resultado else 0,
            'nodes_expanded'     : nodes_expanded,
            'search_depth'       : resultado.profundidad if resultado else 0,
            'max_search_depth'   : max_search_depth,
            'running_time'       : f'{running_time} s',
            'max_ram_usage_mb'   : f'{max_ram} MB',
        }
        m.update(extra)
        return m


print('Clase base Agente cargada OK')


### 2.1 DFS — Depth-First Search
**Estructura:** pila explícita · **Expansión:** UDLR inverso al push


In [ ]:
class DFS(Agente):
    """Agente de búsqueda en profundidad (pila explícita)."""

    nombre = 'DFS'

    def resolver(self, tablero_inicial, output_path='outputs/output_dfs.txt'):
        t_inicio         = self._iniciar_medicion()
        nodes_expanded   = 0
        max_search_depth = 0
        visitados        = set()
        pila             = [Nodo(tablero_inicial.clonar(), [], 0)]
        resultado        = None

        while pila:
            nodo   = pila.pop()                              # 1. ELIMINAR
            estado = nodo.tablero.estado()
            if estado in visitados:
                continue
            visitados.add(estado)
            nodes_expanded  += 1
            max_search_depth = max(max_search_depth, nodo.profundidad)

            if nodo.tablero.esta_completo():                 # 2. VERIFICAR
                resultado = nodo
                break
            if nodo.tablero.sin_salida():
                continue

            for mov in reversed(nodo.tablero.movimientos_validos()):  # 3. EXPANDIR
                r, c  = mov
                nuevo = nodo.tablero.clonar()
                nuevo.seleccionar(r, c)
                if nuevo.estado() not in visitados:
                    pila.append(Nodo(nuevo, nodo.camino + [(r, c)], nodo.profundidad + 1))

        running_time, max_ram = self._terminar_medicion(t_inicio)
        m = self._construir_resultado(resultado, nodes_expanded, max_search_depth, running_time, max_ram)
        self._escribir_output(m, output_path)
        return m


print('DFS cargado OK')


### 2.2 BFS — Breadth-First Search
**Estructura:** cola explícita (deque) · **Expansión:** UDLR


In [ ]:
class BFS(Agente):
    """Agente de búsqueda en anchura (cola explícita)."""

    nombre = 'BFS'

    def resolver(self, tablero_inicial, output_path='outputs/output_bfs.txt'):
        t_inicio         = self._iniciar_medicion()
        nodes_expanded   = 0
        max_search_depth = 0
        visitados        = set()
        cola             = deque([Nodo(tablero_inicial.clonar(), [], 0)])
        resultado        = None

        while cola:
            nodo   = cola.popleft()                          # 1. ELIMINAR
            estado = nodo.tablero.estado()
            if estado in visitados:
                continue
            visitados.add(estado)
            nodes_expanded  += 1
            max_search_depth = max(max_search_depth, nodo.profundidad)

            if nodo.tablero.esta_completo():                 # 2. VERIFICAR
                resultado = nodo
                break
            if nodo.tablero.sin_salida():
                continue

            for mov in nodo.tablero.movimientos_validos():   # 3. EXPANDIR
                r, c  = mov
                nuevo = nodo.tablero.clonar()
                nuevo.seleccionar(r, c)
                if nuevo.estado() not in visitados:
                    cola.append(Nodo(nuevo, nodo.camino + [(r, c)], nodo.profundidad + 1))

        running_time, max_ram = self._terminar_medicion(t_inicio)
        m = self._construir_resultado(resultado, nodes_expanded, max_search_depth, running_time, max_ram)
        self._escribir_output(m, output_path)
        return m


print('BFS cargado OK')


### 2.3 A\* — A-Star Search
**Estructura:** cola de prioridad (heapq) · **f(n) = g(n) + h(n)**

| # | Heurística | Descripción |
|---|---|---|
| H1 | Celdas restantes | Fracción sin visitar |
| H2 | Pasos pendientes | Urgencia del tramo actual |
| H3 | Números restantes | Anclajes sin activar |
| H4 | Aislamiento | Vacías sin número adyacente |
| H5 | Distancia Manhattan | Al número no visitado más cercano |


In [ ]:
class AStarAgent(Agente):
    """Agente A* con 5 heurísticas normalizadas y ponderadas."""

    nombre = 'A*'

    # Configuraciones de pesos (w1..w5) para cada heurística
    CONFIGS = {
        'iguales'   : (0.20, 0.20, 0.20, 0.20, 0.20),
        'config_a'  : (0.35, 0.10, 0.30, 0.15, 0.10),
        'config_b'  : (0.10, 0.25, 0.15, 0.35, 0.15),
        'h1'        : (1.00, 0.00, 0.00, 0.00, 0.00),
        'h1h2'      : (0.50, 0.50, 0.00, 0.00, 0.00),
        'h1h2h3'    : (0.33, 0.33, 0.34, 0.00, 0.00),
        'h1h2h3h4'  : (0.25, 0.25, 0.25, 0.25, 0.00),
        'h1h2h3h4h5': (0.20, 0.20, 0.20, 0.20, 0.20),
    }

    # ── Heurísticas individuales (cada una en [0, 1]) ─────────────────────
    def _h1(self, t):
        return (t.total - t.visitadas) / t.total

    def _h2(self, t):
        max_val = max((t.celdas[r][c].valor
                       for r in range(t.filas) for c in range(t.cols)
                       if t.celdas[r][c].es_numero()), default=1)
        return min(t.pasos / max_val, 1.0)   # clamp a [0,1]

    def _h3(self, t):
        total = sum(1 for r in range(t.filas) for c in range(t.cols)
                    if t.celdas[r][c].es_numero())
        if total == 0: return 0.0
        rest = sum(1 for r in range(t.filas) for c in range(t.cols)
                   if t.celdas[r][c].es_numero() and not t.celdas[r][c].visitada)
        return rest / total

    def _h4(self, t):
        vacias = total = 0
        for r in range(t.filas):
            for c in range(t.cols):
                celda = t.celdas[r][c]
                if celda.es_vacia() and not celda.visitada:
                    total += 1
                    tiene_num = any(t.celdas[nr][nc].es_numero()
                                    and not t.celdas[nr][nc].visitada
                                    for nr, nc in t.adyacentes(r, c))
                    if not tiene_num:
                        vacias += 1
        return vacias / total if total else 0.0

    def _h5(self, t):
        if t.pos is None: return 0.0
        pr, pc   = t.pos
        max_dist = t.filas + t.cols - 2
        if max_dist == 0: return 0.0
        min_dist = min((abs(r-pr) + abs(c-pc)
                        for r in range(t.filas) for c in range(t.cols)
                        if t.celdas[r][c].es_numero() and not t.celdas[r][c].visitada),
                       default=0)
        return min_dist / max_dist

    def _heuristica(self, t, pesos):
        w1, w2, w3, w4, w5 = pesos
        h = 0.0
        if w1: h += w1 * self._h1(t)
        if w2: h += w2 * self._h2(t)
        if w3: h += w3 * self._h3(t)
        if w4: h += w4 * self._h4(t)
        if w5: h += w5 * self._h5(t)
        return h

    def resolver(self, tablero_inicial, config='iguales', output_path='outputs/output_astar.txt'):
        pesos            = self.CONFIGS[config]
        t_inicio         = self._iniciar_medicion()
        nodes_expanded   = 0
        max_search_depth = 0
        contador_fifo    = 0
        visitados        = set()

        h0        = self._heuristica(tablero_inicial, pesos)
        nodo_raiz = NodoAStar(tablero_inicial.clonar(), [], 0, 0.0, h0)
        heap      = []
        heapq.heappush(heap, (nodo_raiz.f, contador_fifo, nodo_raiz))
        resultado = None

        while heap:
            _, _, nodo = heapq.heappop(heap)                 # 1. ELIMINAR
            estado     = nodo.tablero.estado()
            if estado in visitados:
                continue
            visitados.add(estado)
            nodes_expanded  += 1
            max_search_depth = max(max_search_depth, nodo.profundidad)

            if nodo.tablero.esta_completo():                 # 2. VERIFICAR
                resultado = nodo
                break
            if nodo.tablero.sin_salida():
                continue

            for mov in nodo.tablero.movimientos_validos():   # 3. EXPANDIR
                r, c  = mov
                nuevo = nodo.tablero.clonar()
                nuevo.seleccionar(r, c)
                if nuevo.estado() not in visitados:
                    nuevo_g    = nodo.g + 1
                    nuevo_h    = self._heuristica(nuevo, pesos)
                    nodo_nuevo = NodoAStar(nuevo, nodo.camino + [(r, c)],
                                          nodo.profundidad + 1, nuevo_g, nuevo_h)
                    contador_fifo += 1
                    heapq.heappush(heap, (nodo_nuevo.f, contador_fifo, nodo_nuevo))

        running_time, max_ram = self._terminar_medicion(t_inicio)
        m = self._construir_resultado(resultado, nodes_expanded, max_search_depth,
                                      running_time, max_ram, config=config, pesos=pesos)
        self._escribir_output(m, output_path)
        return m


print('A* cargado OK')


## 3. Menú principal


In [ ]:
import re
from google.colab import output as colab_output


class Menu:
    """
    Punto de entrada unificado.
    Flujo: elegir nivel → jugar / agente → (si agente) cuál → output.txt
    """

    NIVELES_INFO = {
        1:  '3x3  — Fácil',        2:  '3x3  — Fácil',
        3:  '4x4  — Fácil',        4:  '4x4  — Fácil',
        5:  '5x5  — Intermedio',   6:  '5x5  — Intermedio',
        7:  '6x6  — Intermedio',   8:  '6x6  — Intermedio',
        9:  '7x7  — Difícil',      10: '7x7  — Difícil',
        11: '8x8  — Difícil',      12: '8x8  — Difícil',
        13: '10x10 — Muy difícil', 14: '10x10 — Muy difícil',
        15: '10x10 — Muy difícil', 16: '10x10 — Muy difícil',
    }

    AGENTES = {'1': 'DFS', '2': 'BFS', '3': 'A*'}

    CONFIGS_ASTAR = {
        '1': 'iguales',     '2': 'config_a',      '3': 'config_b',
        '4': 'h1',          '5': 'h1h2',           '6': 'h1h2h3',
        '7': 'h1h2h3h4',    '8': 'h1h2h3h4h5',
    }

    def __limpiar(self):
        colab_output.clear()

    def __parsear_coords(self, entrada):
        """Acepta: '0 0', '0,0', '0.0'. Evita formato compacto por ambigüedad en 10x10."""
        partes = re.split(r'[^0-9]+', entrada.strip())
        partes = [p for p in partes if p]
        if len(partes) == 2:
            return int(partes[0]), int(partes[1])
        raise ValueError('Formato no reconocido. Usa: fila col  (ej: 0 0 | 0,0 | 0.0)')

    def __elegir_nivel(self):
        while True:
            try:
                self.__limpiar()
                print('=' * 44)
                print('            NUMARI — NIVELES')
                print('=' * 44)
                for n, desc in self.NIVELES_INFO.items():
                    print(f'  {n:>2}. {desc}')
                print('   0. Cargar nivel desde archivo .txt')
                print('-' * 44)
                entrada = input('Elige un nivel: ').strip()
                if entrada == '0':
                    return self.__cargar_txt()
                nivel = int(entrada)
                if nivel not in self.NIVELES_INFO:
                    print('Nivel no existe.')
                    input('[Enter]')
                    continue
                return nivel, f'niveles/nivel_{nivel:02d}.txt'
            except ValueError:
                print('Ingresa un número válido.')
                input('[Enter]')

    def __cargar_txt(self):
        """Permite cargar un nivel personalizado desde cualquier ruta .txt."""
        while True:
            self.__limpiar()
            print('=' * 44)
            print('        CARGAR NIVEL PERSONALIZADO')
            print('=' * 44)
            print('  Formato esperado en el .txt:')
            print('    . 3 . .      (punto = vacía, número = pasos)')
            print('    3 . . .')
            print('    . . . 3')
            print('    . . 3 .')
            print('-' * 44)
            ruta = input('Ruta del archivo (o Enter para volver): ').strip()
            if not ruta:
                return None, None
            try:
                tablero_prueba = Tablero.desde_archivo(ruta)
                print(f'  Nivel cargado: {tablero_prueba.filas}x{tablero_prueba.cols}')
                tablero_prueba.mostrar()
                input('[Enter para continuar]')
                return 'custom', ruta
            except FileNotFoundError:
                print(f'  Archivo no encontrado: {ruta}')
                input('[Enter]')
            except Exception as e:
                print(f'  Error al leer el archivo: {e}')
                input('[Enter]')

    def __elegir_modo(self, nivel, tablero):
        while True:
            self.__limpiar()
            label = self.NIVELES_INFO.get(nivel, 'Personalizado')
            print('=' * 44)
            print(f'       NIVEL {nivel} — {label}')
            print('=' * 44)
            tablero.mostrar()
            print('  1. Jugar manualmente')
            print('  2. Usar agente de búsqueda')
            print('  0. Volver a elegir nivel')
            print('-' * 44)
            op = input('Opción: ').strip()
            if op == '1': return 'jugar'
            if op == '2': return 'agente'
            if op == '0': return 'volver'
            print('Opción inválida.')
            input('[Enter]')

    def __elegir_agente(self, nivel):
        while True:
            self.__limpiar()
            print('=' * 44)
            print(f'       NIVEL {nivel} — ELEGIR AGENTE')
            print('=' * 44)
            for k, v in self.AGENTES.items():
                print(f'  {k}. {v}')
            print('  0. Volver')
            print('-' * 44)
            op = input('Opción: ').strip()
            if op == '0': return None
            if op in self.AGENTES: return self.AGENTES[op]
            print('Opción inválida.')
            input('[Enter]')

    def __elegir_config_astar(self, nivel):
        while True:
            self.__limpiar()
            print('=' * 44)
            print(f'       NIVEL {nivel} — CONFIGURACIÓN A*')
            print('=' * 44)
            agente_tmp = AStarAgent()
            for k, v in self.CONFIGS_ASTAR.items():
                print(f'  {k}. {v:<14} {agente_tmp.CONFIGS[v]}')
            print('  0. Volver')
            print('-' * 44)
            op = input('Opción: ').strip()
            if op == '0': return None
            if op in self.CONFIGS_ASTAR: return self.CONFIGS_ASTAR[op]
            print('Opción inválida.')
            input('[Enter]')

    def __ejecutar_agente(self, agente_nombre, config, tablero, nivel, ruta_txt):
        self.__limpiar()
        cfg_str = config or ''
        print(f'  Ejecutando {agente_nombre} {cfg_str} sobre nivel {nivel}...')
        print('  (Esto puede tardar en niveles grandes)')

        nivel_str = str(nivel) if nivel == 'custom' else f'{nivel:02d}'
        ruta_out  = f'outputs/{agente_nombre.lower()}_{cfg_str}_nivel_{nivel_str}.txt'

        if agente_nombre == 'DFS':
            m = DFS().resolver(tablero, ruta_out)
        elif agente_nombre == 'BFS':
            m = BFS().resolver(tablero, ruta_out)
        else:
            m = AStarAgent().resolver(tablero, config, ruta_out)

        self.__limpiar()
        print('=' * 44)
        print(f'  RESULTADO — {agente_nombre} {cfg_str}')
        print('=' * 44)
        t_res = tablero.clonar()
        if m['solucion_encontrada']:
            for r, c in m['path_to_goal']:
                t_res.seleccionar(r, c)
        t_res.mostrar()
        print(f"  Solución encontrada : {m['solucion_encontrada']}")
        if m['path_to_goal']:
            path_str = ' -> '.join(f'({r},{c})' for r, c in m['path_to_goal'])
            print(f"  Camino              : {path_str}")
        print(f"  Movimientos         : {m['cost_of_path']}")
        print(f"  Nodos expandidos    : {m['nodes_expanded']}")
        print(f"  Profundidad         : {m['search_depth']}")
        print(f"  Prof. máx búsqueda  : {m['max_search_depth']}")
        print(f"  Tiempo              : {m['running_time']}")
        print(f"  RAM máx             : {m['max_ram_usage_mb']}")
        print(f"  Output guardado en  : {ruta_out}")
        print('=' * 44)
        input('\n[Enter para continuar]')

    def __jugar(self, nivel, tablero, ruta_txt):
        msg = ''
        while True:
            self.__limpiar()
            label = self.NIVELES_INFO.get(nivel, 'Personalizado')
            print('=' * 44)
            print(f'         NUMARI — NIVEL {nivel} ({label})')
            print('=' * 44)
            print('  fila col → Seleccionar  (ej: 0 0 | 0,0 | 0.0)')
            print('  R        → Reiniciar')
            print('  Q        → Volver al menú')
            print('-' * 44)
            tablero.mostrar()
            if msg:
                print(f'  >> {msg}')
            entrada = input('\n>> ').strip().upper()
            msg = ''
            if not entrada: continue
            if entrada == 'Q': return
            if entrada == 'R':
                tablero = Tablero.desde_archivo(ruta_txt)
                msg = 'Tablero reiniciado.'
                continue
            try:
                fila, col = self.__parsear_coords(entrada)
                msg = tablero.seleccionar(fila, col)
                if tablero.esta_completo():
                    self.__limpiar()
                    print('=' * 44)
                    print('          FELICIDADES! GANASTE!')
                    print('=' * 44)
                    tablero.mostrar()
                    input('\n[Enter para volver al menú]')
                    return
                if tablero.sin_salida():
                    msg = 'Sin salida. Reinicia con R.'
            except ValueError as e:
                msg = f'(!!) {e}'

    def init(self):
        while True:
            nivel, ruta_txt = self.__elegir_nivel()
            if nivel is None:
                continue
            tablero = Tablero.desde_archivo(ruta_txt)
            modo    = self.__elegir_modo(nivel, tablero)
            if modo == 'volver': continue
            if modo == 'jugar':
                self.__jugar(nivel, tablero, ruta_txt)
                continue
            agente = self.__elegir_agente(nivel)
            if agente is None: continue
            config = None
            if agente == 'A*':
                config = self.__elegir_config_astar(nivel)
                if config is None: continue
            tablero = Tablero.desde_archivo(ruta_txt)
            self.__ejecutar_agente(agente, config, tablero, nivel, ruta_txt)


print('Menu cargado OK')


## 4. Iniciar


In [ ]:
dependencias = {
    'Tablero'    : 'Tablero'    in dir(),
    'Agente'     : 'Agente'     in dir(),
    'DFS'        : 'DFS'        in dir(),
    'BFS'        : 'BFS'        in dir(),
    'AStarAgent' : 'AStarAgent' in dir(),
    'Menu'       : 'Menu'       in dir(),
}
faltantes = [k for k, v in dependencias.items() if not v]
if faltantes:
    print('(!!) Ejecuta primero todas las celdas anteriores en orden.')
    print(f'     Faltan: {faltantes}')
else:
    menu = Menu()
    menu.init()
